In [ ]:
#note:done w/ claude

In [1]:
import requests
import pandas as pd
import time

print("Starting historical extraction for 2026 from OpenStreetMap...")

# UPDATED: Just asking for 2026
years_to_fetch = [2026] 
all_stations = []
overpass_url = "http://overpass-api.de/api/interpreter"

headers = {
    'User-Agent': 'UKCrimeDataProject/1.0 (Contact: catherinekasia@gmail.com)'
}

uk_regions = {
    "Scotland": "54.6,-8.0,60.9,0.0",
    "North England": "53.0,-4.0,54.6,0.0",
    "Wales & West Midlands": "51.4,-5.5,53.0,-1.0",
    "East Midlands & East Anglia": "51.4,-1.0,53.0,2.0",
    "South England": "49.9,-6.0,51.4,2.0",
    "Northern Ireland": "54.0,-8.5,55.3,-5.0"
}

for year in years_to_fetch:
    print(f"\n--- Fetching Data for {year} ---")
    for region_name, bbox in uk_regions.items():
        print(f"  -> Querying {region_name}...")
        
        # The API will pull the map exactly as it existed on 2026-01-01
        overpass_query = f"""
        [out:json][timeout:180][date:"{year}-01-01T00:00:00Z"][bbox:{bbox}];
        (
          node["amenity"="police"];
          way["amenity"="police"];
          relation["amenity"="police"];
        );
        out center;
        """
        
        try:
            response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers)
            response.raise_for_status() 
            data = response.json()
            
            for element in data.get('elements', []):
                lat = element.get('lat', element.get('center', {}).get('lat'))
                lon = element.get('lon', element.get('center', {}).get('lon'))
                
                if lat and lon:
                    all_stations.append({
                        'year': year,
                        'latitude': lat,
                        'longitude': lon,
                        'name': element.get('tags', {}).get('name', 'Unknown Station')
                    })
                    
            time.sleep(8) 
            
        except Exception as e:
            print(f"  [!] Failed to fetch {region_name}: {e}")

stations_df = pd.DataFrame(all_stations)
stations_df = stations_df.drop_duplicates(subset=['year', 'latitude', 'longitude'])

print(f"\nSuccess! Found {len(stations_df)} station records for 2026 across the UK.")
display(stations_df.head(10))

/Users/catherinekmiec/miniforge3/envs/4CBL_native/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Starting historical extraction for 2026 from OpenStreetMap...

--- Fetching Data for 2026 ---
  -> Querying Scotland...
  -> Querying North England...
  -> Querying Wales & West Midlands...
  -> Querying East Midlands & East Anglia...
  -> Querying South England...
  -> Querying Northern Ireland...

Success! Found 1959 station records for 2026 across the UK.


,year,latitude,longitude,name
0,2026,55.750002,-4.936241,Unknown Station
1,2026,55.860672,-4.027778,Coatbridge Police station
2,2026,55.799724,-4.390692,Barrhead Police Station
3,2026,56.036071,-5.431863,Unknown Station
4,2026,56.467454,-2.877516,Broughty Ferry Police Station
5,2026,57.583259,-4.128058,Unknown Station
6,2026,56.481861,-3.450400,Unknown Station
7,2026,54.971733,-2.461151,Haltwhistle Police House
8,2026,55.778664,-2.345754,Duns Police Office
9,2026,57.513373,-4.456678,Muir of Ord Police Station and Council Service...


In [2]:
import folium
from folium.plugins import MarkerCluster

print("Building the interactive UK map...")

# 1. Create a base map centered roughly in the middle of the UK
uk_map = folium.Map(location=[54.0, -2.5], zoom_start=6)

# 2. Create a 'Cluster' layer (groups points together when zoomed out)
marker_cluster = MarkerCluster().add_to(uk_map)

# 3. Drop every single police station onto the map
for idx, row in stations_df.iterrows():
    # We add a little popup tag so you can click the pin to see the station's name!
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=row['name'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(marker_cluster)

# 4. Save it as a webpage
map_filename = "uk_police_stations_2026.html"
uk_map.save(map_filename)

print(f"Success! Open the file '{map_filename}' in your web browser to explore.")

Building the interactive UK map...
Success! Open the file 'uk_police_stations_2026.html' in your web browser to explore.


In [11]:
print("--- Geographic Boundaries ---")
print(f"Furthest North (Latitude): {stations_df['latitude'].max():.4f}  (Should be ~60.0 for Scotland)")
print(f"Furthest South (Latitude): {stations_df['latitude'].min():.4f}  (Should be ~50.0 for Cornwall)")
print(f"Furthest East (Longitude): {stations_df['longitude'].max():.4f}  (Should be ~1.7 for Norfolk)")
print(f"Furthest West (Longitude): {stations_df['longitude'].min():.4f}  (Should be ~-8.0 for N. Ireland)")

# Let's also see how many stations don't have a formal name attached
unknowns = len(stations_df[stations_df['name'] == 'Unknown Station'])
print(f"\nStations without a formal name: {unknowns} out of {len(stations_df)}")

--- Geographic Boundaries ---
Furthest North (Latitude): 60.7627  (Should be ~60.0 for Scotland)
Furthest South (Latitude): 49.9008  (Should be ~50.0 for Cornwall)
Furthest East (Longitude): 1.9835  (Should be ~1.7 for Norfolk)
Furthest West (Longitude): -8.4884  (Should be ~-8.0 for N. Ireland)

Stations without a formal name: 500 out of 1959


In [4]:
import geopandas as gpd
import sqlite3

# 1. Load your LSOA map (The "Filter")
# Replace with your actual path to the LSOA GeoJSON or Shapefile
print("Loading LSOA boundaries...")
lsoa_map = gpd.read_file('../data/lsoa_spatial.geojson') 

# 2. Convert your stations to a GeoDataFrame
print("Converting station data to spatial format...")
stations_gdf = gpd.GeoDataFrame(
    stations_df, 
    geometry=gpd.points_from_xy(stations_df.longitude, stations_df.latitude),
    crs="EPSG:4326"
)

# Ensure the coordinate systems match
if stations_gdf.crs != lsoa_map.crs:
    stations_gdf = stations_gdf.to_crs(lsoa_map.crs)

# 3. THE SPATIAL JOIN (This automatically deletes the French stations!)
print("Filtering stations through the LSOA map...")
stations_in_uk = gpd.sjoin(stations_gdf, lsoa_map, how="inner", predicate="within")

# 4. Count them up
# Change 'LSOA21CD' to the column name for the LSOA code in your geojson file
lsoa_counts = stations_in_uk.groupby('LSOA21CD').size().reset_index(name='police_station_count')

print(f"\nCleaned! We went from {len(stations_df)} raw points down to {len(stations_in_uk)} UK-verified stations.")
print(f"Number of LSOAs with at least one station: {len(lsoa_counts)}")

# 5. Save to your SQL Database
conn = sqlite3.connect('../data/police_data.db')
lsoa_counts.to_sql('lsoa_infrastructure', conn, if_exists='replace', index=False)
conn.close()

print("\nSuccess! The cleaned 'police_station_count' is now in your database.")

Loading LSOA boundaries...
Converting station data to spatial format...
Filtering stations through the LSOA map...

Cleaned! We went from 1959 raw points down to 1477 UK-verified stations.
Number of LSOAs with at least one station: 1369

Success! The cleaned 'police_station_count' is now in your database.


In [6]:
import sqlite3

conn = sqlite3.connect('../data/police_data.db')

conn.execute('ALTER TABLE lsoa_infrastructure RENAME COLUMN LSOA21CD TO lsoa_code')
conn.commit()
conn.close()

print("Done! LSOA21CD renamed to lsoa_code in lsoa_infrastructure.")

OperationalError: no such column: "LSOA21CD"